# COVID-19 and the economy: exploratory data analysis

**Seven countries (Brazil, China, France, Germany, India, Rwanda, United States) · economy 2018-2024 · health and policy 2020-2022**

This notebook is the analysis deliverable. It contains summary-statistics tables, charts with an explanation of what each one reveals, evidence-based findings for five dashboard themes, and an automatically written short findings document for the final presentation.

| Theme | Question | Data used |
|---|---|---|
| 1. Health burden | How hard and when did the virus hit? | monthly and yearly health data |
| 2. Government response | How strict were restrictions, and for whom? | monthly stringency index |
| 3. Economic output and recovery | How large was the shock, and did economies recover? | GDP growth 2018-2024 |
| 4. Jobs and prices | How did unemployment and inflation move? | unemployment and inflation 2018-2024 |
| 5. Linking health, policy and the economy | Does the economic loss follow deaths or strictness? | all files combined |

**To run it:** upload the five CSV files to the same place as the notebook (in Colab: folder icon in the left sidebar, then upload) and run the cells from top to bottom. Every number in the findings sections is computed from the data, so nothing is typed by hand.

## 1. Setup and visual style

In [1]:
import os, glob
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import spearmanr
from IPython.display import display, Markdown

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 30)

# ---- Visual identity: one palette used by every chart ----
INK, MUTED, GRID = "#2B2D42", "#6B7280", "#E5E7EB"
COUNTRY_COLORS = {            # colour-blind-friendly, one fixed colour per country
    "Brazil": "#009E73", "China": "#D55E00", "France": "#0072B2", "Germany": "#4D4D4D",
    "India": "#E69F00", "Rwanda": "#CC79A7", "United States": "#56B4E9"}
YEAR_COLORS = {"2020": "#C0392B", "2021": "#2E86AB", "2022": "#8D99AE"}
IND_COLORS  = {"GDP growth (%)": "#0072B2", "Inflation (%)": "#D55E00", "Unemployment (%)": "#009E73"}
SHADE = "rgba(192, 57, 43, 0.10)"          # shading used for the COVID years
COUNTRIES = list(COUNTRY_COLORS)

FIGS = {}   # every chart is stored here and exported at the end

def polish(fig, title, subtitle=None, height=520, legend=True, facets=False):
    """Apply the same clean look to every chart."""
    t = f"<b>{title}</b>"
    if subtitle:
        t += f"<br><span style='font-size:13px;color:{MUTED}'>{subtitle}</span>"
    fig.update_layout(
        template="plotly_white", height=height,
        font=dict(family="Arial, sans-serif", size=13, color=INK),
        title=dict(text=t, x=0, xanchor="left"),
        margin=dict(t=115, l=60, r=30, b=60),
        paper_bgcolor="white", plot_bgcolor="white",
        showlegend=legend,
        legend=dict(orientation="h", y=1.02, yanchor="bottom", x=0, title_text=""),
        hoverlabel=dict(font_size=12))
    fig.update_xaxes(showgrid=False, linecolor="#9CA3AF")
    fig.update_yaxes(gridcolor=GRID, zerolinecolor="#9CA3AF")
    if facets:
        fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font=dict(size=13, color=INK)))
    return fig

def style_table(df, gradients=None, formats=None):
    """Readable, lightly coloured tables."""
    s = df.style.format(formats or "{:,.1f}")
    for cols, cmap in (gradients or []):
        s = s.background_gradient(subset=cols, cmap=cmap)
    return s.set_table_styles([{"selector": "th", "props": [("background-color", "#F3F4F6"), ("color", INK), ("font-weight", "600")]}])

## 2. Loading the data

Two of the file families use different separators (`,` or `;`), so the loader reads the first line of each file and chooses the separator itself. Files are found by keyword, so small differences in file names (for example `inflation_pct_annual (1).csv`) do not matter.

In [7]:
from google.colab import files
uploaded = files.upload()

Saving merged_dataset_by_year.csv to merged_dataset_by_year.csv
Saving unemployment_pct_annual.csv to unemployment_pct_annual.csv
Saving gdp_growth_pct_annual.csv to gdp_growth_pct_annual.csv
Saving inflation_pct_annual.csv to inflation_pct_annual.csv
Saving merged_dataset_by_month.csv to merged_dataset_by_month.csv


In [8]:




gdp   = pd.read_csv("gdp_growth_pct_annual.csv",    sep=",")
infl  = pd.read_csv("inflation_pct_annual.csv", sep=";")
unemp = pd.read_csv("unemployment_pct_annual.csv",  sep=";")
y = pd.read_csv("merged_dataset_by_year.csv",  sep=";")
m = pd.read_csv("merged_dataset_by_month.csv", sep=";")

# 2) Economy table, 2018-2024 (49 rows = 7 countries x 7 years)
eco = (gdp.merge(infl, on=["country", "code", "year"])
          .merge(unemp, on=["country", "code", "year"])
          .sort_values(["country", "year"]).reset_index(drop=True))

# 3) Small preparations
m["date"] = pd.to_datetime(m["month"])
y["year_str"] = y["year"].astype(str)

# 4) Master table for Act 3: economy + health/policy + pre-COVID baseline
health_cols = ["new_cases_per_million", "new_deaths_per_million",
               "sti_vaccinated", "sti_nonvaccinated", "sti_weighted"]
full = eco.merge(y[["country", "code", "year"] + health_cols],
                 on=["country", "code", "year"], how="left")   # health is empty outside 2020-2022
base = (eco[eco.year.isin([2018, 2019])]
        .groupby("country")[["gdp_growth_pct", "inflation_pct", "unemployment_pct"]]
        .mean().add_suffix("_base").reset_index())
full = full.merge(base, on="country")
full["gdp_effect"]   = full["gdp_growth_pct"]   - full["gdp_growth_pct_base"]    # points below/above own normal
full["unemp_effect"] = full["unemployment_pct"] - full["unemployment_pct_base"]
full["infl_effect"]  = full["inflation_pct"]    - full["inflation_pct_base"]

cov = full[full.year.between(2020, 2022)].copy()               # the 21 rows where all data exist
cov["year_str"] = cov["year"].astype(str)

print(eco.shape, y.shape, m.shape, cov.shape)   # (49, 6) (21, 12) (252, 12) (21, 18)

(49, 6) (21, 12) (252, 12) (21, 18)


## 3. Data checks

In [9]:
overview = pd.DataFrame({
    "File": ["GDP growth", "Inflation", "Unemployment", "Health + policy + economy (yearly)", "Health + policy + economy (monthly)"],
    "Rows": [len(gdp), len(infl), len(unemp), len(y), len(m)],
    "Countries": [d["country"].nunique() for d in (gdp, infl, unemp, y, m)],
    "First period": [d["year"].min() if "year" in d else d["month"].min() for d in (gdp, infl, unemp, y, m)],
    "Last period": [d["year"].max() if "year" in d else d["month"].max() for d in (gdp, infl, unemp, y, m)],
    "Missing cells": [int(d.isna().sum().sum()) for d in (gdp, infl, unemp, y, m)]})
overview[["First period", "Last period"]] = overview[["First period", "Last period"]].astype(str)
display(style_table(overview, formats={"Rows": "{:,.0f}", "Countries": "{:,.0f}", "Missing cells": "{:,.0f}"}).hide(axis="index"))

# Do the monthly and yearly files agree? Aggregate months to years and compare.
m["date"] = pd.to_datetime(m["month"])
m["year"] = m["date"].dt.year
chk = (m.groupby(["country", "year"]).agg(cases=("new_cases_per_million", "sum"),
                                          deaths=("new_deaths_per_million", "sum"),
                                          sti=("sti_weighted", "mean")).reset_index()
         .merge(y, on=["country", "year"]))
print("Largest difference between the two files  ->  cases:", round((chk.cases - chk.new_cases_per_million).abs().max(), 3),
      "| deaths:", round((chk.deaths - chk.new_deaths_per_million).abs().max(), 3),
      "| stringency:", round((chk.sti - chk.sti_weighted).abs().max(), 3))
print("Economic values per country-year in the monthly file (1 = repeated, not measured monthly):",
      m.groupby(["country", "year"])[["gdp_growth_pct", "inflation_pct", "unemployment_pct"]].nunique().max().max())

File,Rows,Countries,First period,Last period,Missing cells
GDP growth,49,7,2018,2024,0
Inflation,49,7,2018,2024,0
Unemployment,49,7,2018,2024,0
Health + policy + economy (yearly),21,7,2020,2022,0
Health + policy + economy (monthly),252,7,2020-01,2022-12,0


Largest difference between the two files  ->  cases: 0.0 | deaths: 0.0 | stringency: 0.005
Economic values per country-year in the monthly file (1 = repeated, not measured monthly): 1


The monthly file is a finer version of the yearly one: cases and deaths add up exactly, and stringency averages out. Its economic columns are the yearly value repeated twelve times, so the **monthly file is used only for the shape of the epidemic and of policy over time**, and the **yearly file for every comparison with the economy**.

## 4. Preparing the analysis tables

- `eco`: economy, 2018-2024 (49 rows).
- `full` / `cov`: economy joined with health and policy for 2020-2022 (21 rows), with each country's own **2018-2019 average** as the pre-COVID baseline. `gdp_effect` is growth minus that baseline, which is fairer than raw growth because it compares each country with its own normal.
- `idx`: size of each economy with 2019 = 100, and the size it would have reached had 2018-2019 growth continued (an estimate).

In [10]:
y["year_str"] = y["year"].astype(str)
m["sti_gap"] = m["sti_nonvaccinated"] - m["sti_vaccinated"]

ECON = {"gdp_growth_pct": "GDP growth (%)", "inflation_pct": "Inflation (%)", "unemployment_pct": "Unemployment (%)"}
eco = (gdp.merge(infl, on=["country", "code", "year"])
          .merge(unemp, on=["country", "code", "year"])
          .sort_values(["country", "year"]).reset_index(drop=True))

def period_of(yr):
    if yr <= 2019: return "2018-19 (before COVID)"
    if yr <= 2021: return "2020-21 (shock)"
    if yr == 2022: return "2022"
    return "2023-24 (after)"
PERIODS = ["2018-19 (before COVID)", "2020-21 (shock)", "2022", "2023-24 (after)"]
eco["period"] = eco["year"].map(period_of)

# --- economy + health/policy + baseline effects ---
health_cols = ["new_cases_per_million", "new_deaths_per_million", "sti_vaccinated", "sti_nonvaccinated", "sti_weighted"]
full = eco.merge(y[["country", "code", "year"] + health_cols], on=["country", "code", "year"], how="left")
base = (eco[eco.year.isin([2018, 2019])].groupby("country")[list(ECON)].mean().add_suffix("_base").reset_index())
full = full.merge(base, on="country")
full["gdp_effect"]   = full["gdp_growth_pct"]   - full["gdp_growth_pct_base"]
full["unemp_effect"] = full["unemployment_pct"] - full["unemployment_pct_base"]
full["infl_effect"]  = full["inflation_pct"]    - full["inflation_pct_base"]
cov = full[full.year.between(2020, 2022)].copy()
cov["year_str"] = cov["year"].astype(str)

# --- size of the economy (2019 = 100) and the pre-COVID trend ---
idx = eco[eco.year >= 2019].copy()
idx["factor"] = 1 + idx["gdp_growth_pct"] / 100
idx.loc[idx.year == 2019, "factor"] = 1.0
idx["gdp_index"] = 100 * idx.groupby("country")["factor"].cumprod()
trend_growth = eco[eco.year.isin([2018, 2019])].groupby("country")["gdp_growth_pct"].mean().rename("trend_growth")
idx = idx.merge(trend_growth, on="country")
idx["trend_index"] = 100 * (1 + idx["trend_growth"] / 100) ** (idx["year"] - 2019)
idx["gap_pct"] = idx["gdp_index"] / idx["trend_index"] * 100 - 100

print(eco.shape, y.shape, m.shape, cov.shape, idx.shape)

(49, 7) (21, 12) (252, 14) (21, 19) (42, 12)


## 5. Summary-statistics tables

**Table 1. The typical economy by period** (median across the seven countries).

In [11]:
t1 = (eco.groupby("period")[list(ECON)].median().reindex(PERIODS).rename(columns=ECON))
display(style_table(t1, formats="{:.1f}", gradients=[(["GDP growth (%)"], "RdBu"), (["Inflation (%)"], "Oranges"), (["Unemployment (%)"], "Greens")]))

,GDP growth (%),Inflation (%),Unemployment (%)
period,,,
2018-19 (before COVID),2.8,2.3,7.1
2020-21 (shock),3.1,2.7,7.9
2022,3.0,6.9,5.0
2023-24 (after),3.3,4.2,4.6


**Table 2. Economy by country, 2018-2024.**

In [12]:
rows = []
for c, g in eco.groupby("country"):
    rows.append({"Country": c,
                 "Average GDP growth (%)": g.gdp_growth_pct.mean(),
                 "Worst GDP growth (%)": g.gdp_growth_pct.min(),
                 "Worst year": int(g.loc[g.gdp_growth_pct.idxmin(), "year"]),
                 "Peak inflation (%)": g.inflation_pct.max(),
                 "Peak inflation year": int(g.loc[g.inflation_pct.idxmax(), "year"]),
                 "Peak unemployment (%)": g.unemployment_pct.max(),
                 "Peak unemployment year": int(g.loc[g.unemployment_pct.idxmax(), "year"])})
t2 = pd.DataFrame(rows).set_index("Country")
fmt = {c: "{:.1f}" for c in t2.columns}; fmt.update({c: "{:d}" for c in t2.columns if "year" in c})
display(style_table(t2, formats=fmt, gradients=[(["Worst GDP growth (%)"], "RdBu"), (["Peak inflation (%)"], "Oranges"), (["Peak unemployment (%)"], "Greens")]))

,Average GDP growth (%),Worst GDP growth (%),Worst year,Peak inflation (%),Peak inflation year,Peak unemployment (%),Peak unemployment year
Country,,,,,,,
Brazil,2.0,-3.3,2020,9.3,2022,13.7,2020
China,5.3,2.3,2020,2.9,2019,5.0,2020
France,1.2,-7.4,2020,5.2,2022,9.0,2018
Germany,0.3,-4.1,2020,6.9,2022,3.9,2020
India,5.2,-5.8,2020,6.7,2022,7.9,2020
Rwanda,6.9,-2.8,2020,19.8,2023,15.8,2021
United States,2.6,-2.1,2020,8.0,2022,8.1,2020


**Table 3. Health and policy by country, 2020-2022.**

In [13]:
pk_d = m.loc[m.groupby("country")["new_deaths_per_million"].idxmax()].set_index("country")
pk_s = m.loc[m.groupby("country")["sti_weighted"].idxmax()].set_index("country")
mx_g = m.loc[m.groupby("country")["sti_gap"].idxmax()].set_index("country")
t3 = pd.DataFrame({
    "Reported deaths per million, 2020-22": y.groupby("country")["new_deaths_per_million"].sum(),
    "Reported cases per million, 2020-22":  y.groupby("country")["new_cases_per_million"].sum(),
    "Deadliest month": pk_d["date"].dt.strftime("%b %Y"),
    "Average stringency (0-100)": y.groupby("country")["sti_weighted"].mean(),
    "Strictest month": pk_s["date"].dt.strftime("%b %Y"),
    "Peak stringency": pk_s["sti_weighted"],
    "Largest gap, non-vaccinated vs vaccinated (pts)": mx_g["sti_gap"]})
fmt = {"Reported deaths per million, 2020-22": "{:,.0f}", "Reported cases per million, 2020-22": "{:,.0f}",
       "Average stringency (0-100)": "{:.1f}", "Peak stringency": "{:.1f}",
       "Largest gap, non-vaccinated vs vaccinated (pts)": "{:.1f}"}
display(style_table(t3, formats=fmt, gradients=[(["Reported deaths per million, 2020-22"], "Reds"), (["Average stringency (0-100)"], "YlOrRd")]))

,"Reported deaths per million, 2020-22","Reported cases per million, 2020-22",Deadliest month,Average stringency (0-100),Strictest month,Peak stringency,"Largest gap, non-vaccinated vs vaccinated (pts)"
country,,,,,,,
Brazil,"3,296","172,252",Apr 2021,48.5,May 2020,80.5,23.3
China,31,"52,560",Dec 2022,73.3,Mar 2020,79.5,0.0
France,"2,515","591,025",Nov 2020,42.3,Apr 2020,88.0,45.4
Germany,"1,971","442,902",Dec 2020,44.1,Jan 2021,83.5,23.2
India,372,"31,344",May 2021,57.9,Apr 2020,98.6,20.4
Rwanda,107,"9,741",Jul 2021,53.8,Apr 2020,90.7,44.5
United States,"3,174","290,254",Jan 2021,48.4,Apr 2020,72.7,28.7


**Table 4. Economic impact by country** (all measured against the country's own 2018-2019 level or trend).

In [14]:
u = eco.pivot(index="country", columns="year", values="unemployment_pct")
impact = pd.DataFrame({
    "GDP growth vs own 2018-19 average, 2020 (pts)": full[full.year == 2020].set_index("country")["gdp_effect"],
    "Unemployment change, 2019 to 2020 (pts)": u[2020] - u[2019],
    "Inflation vs own 2018-19 average, 2022 (pts)": full[full.year == 2022].set_index("country")["infl_effect"],
    "Economy vs 2018-19 trend, 2024 (%)": idx[idx.year == 2024].set_index("country")["gap_pct"]})
display(style_table(impact, formats="{:+.1f}", gradients=[
    (["GDP growth vs own 2018-19 average, 2020 (pts)", "Economy vs 2018-19 trend, 2024 (%)"], "RdBu"),
    (["Unemployment change, 2019 to 2020 (pts)", "Inflation vs own 2018-19 average, 2022 (pts)"], "RdBu_r")]))

,"GDP growth vs own 2018-19 average, 2020 (pts)","Unemployment change, 2019 to 2020 (pts)","Inflation vs own 2018-19 average, 2022 (pts)","Economy vs 2018-19 trend, 2024 (%)"
country,,,,
Brazil,-4.8,+1.8,+5.6,+3.4
China,-4.1,+0.4,-0.5,-7.1
France,-9.3,-0.4,+3.7,-4.8
Germany,-5.2,+0.7,+5.3,-5.1
India,-10.9,+1.3,+2.9,-0.7
Rwanda,-11.5,+0.6,+16.2,-11.1
United States,-4.9,+4.4,+5.9,-1.7


---
# Theme 1. Health burden
*How hard did the virus hit each country, and when?*

### Chart 1. Deaths peaked at different times in each country

In [15]:
fig = px.area(m, x="date", y="new_deaths_per_million", color="country", facet_col="country", facet_col_wrap=4,
              color_discrete_map=COUNTRY_COLORS, category_orders={"country": COUNTRIES},
              labels={"new_deaths_per_million": "Deaths per million", "date": ""})
fig.update_yaxes(matches=None, showticklabels=True, title=None)
fig.update_xaxes(title=None, dtick="M12", tickformat="%Y")
fig.update_traces(line_width=1.8)
polish(fig, "Reported COVID-19 deaths per million, month by month",
       "Each panel has its own vertical scale: compare the timing of the waves, not the heights", height=600, legend=False, facets=True)
FIGS["chart_01_deaths_waves"] = fig; fig.show()

**What it reveals.** The waves were not synchronized. France (November 2020, 317 per million) and Germany (December 2020, 278) peaked first, the United States in January 2021 (284), Brazil in April 2021 (400, the highest of all), India in May 2021 (84) and Rwanda in July 2021 (26). China's highest month was December 2022 (10), after its restrictions were lifted. Separate vertical scales are used because death rates differ by orders of magnitude, so the chart supports comparing timing only.

### Chart 2. Reported deaths were concentrated in Brazil and the United States

In [16]:
fig = px.bar(y, x="country", y="new_deaths_per_million", color="year_str", barmode="group",
             color_discrete_map=YEAR_COLORS, category_orders={"country": COUNTRIES},
             labels={"new_deaths_per_million": "Reported deaths per million", "year_str": "Year", "country": ""})
fig.update_traces(texttemplate="%{y:,.0f}", textposition="outside", cliponaxis=False, textfont_size=10)
polish(fig, "Reported COVID-19 deaths per million, by country and year",
       "Reported figures depend on testing and registration rules, so they are not fully comparable across countries", height=500)
FIGS["chart_02_deaths_by_year"] = fig; fig.show()

**What it reveals.** Brazil in 2021 recorded the highest figure (2,034 per million), followed by the United States in 2021 (1,375). France was highest in 2020 (975) and Germany in 2021 (858). India (104, 233 and 35 per million), Rwanda (6, 93 and 9) and China (3, 1 and 27) report far lower figures, which partly reflects differences in testing and reporting and should not be read as a milder epidemic.

---
# Theme 2. Government response
*How strict were restrictions, and did they differ by vaccination status?*

### Chart 3. Restrictions peaked in spring 2020 and eased in 2022, except in China

In [17]:
heat = m.pivot(index="country", columns="month", values="sti_weighted").reindex(COUNTRIES[::-1])
fig = px.imshow(heat, aspect="auto", color_continuous_scale="YlOrRd", zmin=0, zmax=100,
                labels=dict(color="Stringency<br>(0-100)"))
fig.update_xaxes(tickmode="array", tickvals=list(heat.columns[::3]), tickangle=-45, title=None)
fig.update_yaxes(title=None)
polish(fig, "Stringency of government restrictions, month by month",
       "Index from 0 (no restrictions) to 100 (strictest); darker means stricter", height=470, legend=False)
FIGS["chart_03_stringency_heatmap"] = fig; fig.show()

**What it reveals.** Most countries reached their strictest month in spring 2020: India 98.6, Rwanda 90.7, France 88.0, Brazil 80.5, China 79.5 and the United States 72.7. Germany peaked later, at 83.5 in January 2021. By December 2022 France and Germany had fallen to 11.1, while China was still at 60.6, and China's yearly average rose from 68.3 in 2020 to 77.6 in 2022, the only country where restrictions were stricter in 2022 than in 2020.

### Chart 4. Rules for non-vaccinated people became stricter than for vaccinated people

In [18]:
fig = px.line(m, x="date", y="sti_gap", color="country", color_discrete_map=COUNTRY_COLORS,
              category_orders={"country": COUNTRIES},
              labels={"sti_gap": "Index points", "date": ""})
fig.update_traces(line_width=2.6)
fig.update_xaxes(dtick="M6", tickformat="%b %Y")
polish(fig, "Gap in restriction strictness: non-vaccinated minus vaccinated people",
       "Zero means identical rules for both groups", height=500)
FIGS["chart_04_policy_gap"] = fig; fig.show()

**What it reveals.** The gap is zero everywhere in 2020, before vaccines existed. It opened at different dates in 2021: the United States in April, France in June, Germany and India in August, Brazil in September and Rwanda in December. France reached the largest gap in 2021 (45.4 points, September) and Rwanda in 2022 (44.5, October). China shows no gap at any point.

---
# Theme 3. Economic output and recovery
*How large was the shock, and did economies recover?*

### Chart 5. The typical economy: output and jobs fell in 2020, prices rose in 2022

In [19]:
med = eco.groupby("year")[list(ECON)].median().reset_index().melt(id_vars="year", var_name="indicator", value_name="median")
med["indicator"] = med["indicator"].map(ECON)
fig = px.line(med, x="year", y="median", color="indicator", facet_col="indicator", markers=True,
              color_discrete_map=IND_COLORS, labels={"median": "", "year": ""})
fig.update_yaxes(matches=None, showticklabels=True, title=None)
fig.update_xaxes(dtick=2, title=None)
fig.update_traces(line_width=3.2, marker_size=9)
fig.add_vrect(x0=2019.5, x1=2021.5, fillcolor=SHADE, line_width=0, layer="below", row="all", col="all")
polish(fig, "The typical economy, 2018-2024 (median of the seven countries)",
       "Shaded band = 2020-2021", height=430, legend=False, facets=True)
FIGS["chart_05_typical_economy"] = fig; fig.show()

**What it reveals.** Median GDP growth fell from 2.6% in 2019 to -3.3% in 2020 and rebounded to 6.9% in 2021. Median unemployment rose from 6.5% to 8.0% in 2020 and was back to 6.4% in 2021. Median inflation barely moved in 2020 (2.4%) and rose only in 2022 (6.9%), which indicates two separate shocks two years apart. The median is used so that a single extreme country does not distort the picture.

### Chart 6. Six of the seven economies contracted in 2020

In [20]:
piv = eco.pivot(index="country", columns="year", values="gdp_growth_pct").reindex(COUNTRIES[::-1])
piv.columns = piv.columns.astype(str)
fig = px.imshow(piv, text_auto=".1f", aspect="auto", color_continuous_scale="RdBu", color_continuous_midpoint=0,
                labels=dict(color="GDP growth<br>(%)"))
fig.update_xaxes(title=None); fig.update_yaxes(title=None)
polish(fig, "Annual GDP growth by country (%)", "Red = contraction, blue = growth", height=450, legend=False)
FIGS["chart_06_gdp_heatmap"] = fig; fig.show()

**What it reveals.** In 2020 France (-7.4%), India (-5.8%), Germany (-4.1%), Brazil (-3.3%), Rwanda (-2.8%) and the United States (-2.1%) contracted. China grew by 2.3%, far below its 6.1% in 2019. All seven rebounded in 2021. Germany contracted again in 2023 and 2024 (-0.9% and -0.5%), while India stayed above 7%.

### Chart 7. Has each economy regained its 2019 size?

In [21]:
fig = px.line(idx, x="year", y="gdp_index", color="country", markers=True, color_discrete_map=COUNTRY_COLORS,
              category_orders={"country": COUNTRIES},
              labels={"gdp_index": "Size of the economy (2019 = 100)", "year": ""})
fig.update_traces(line_width=2.8, marker_size=8)
fig.add_hline(y=100, line_dash="dash", line_color=INK, annotation_text="2019 level", annotation_position="bottom right")
fig.update_xaxes(dtick=1)
polish(fig, "Size of each economy relative to its own 2019 level",
       "Above the dashed line = larger than in 2019; each country is compared only with itself", height=500)
FIGS["chart_07_economy_index"] = fig; fig.show()

**What it reveals.** Six of the seven economies fell below their 2019 size in 2020, most sharply France (92.6) and India (94.2); China (102.3) did not. France and Germany were still below 2019 in 2021 (98.9 and 99.6), because a rebound applies to a smaller economy. By 2022 all seven were above their 2019 size. By 2024 Rwanda (134.7), India (127.7) and China (126.8) were more than a quarter larger than in 2019, while Germany had only just returned to 100.

### Chart 8. Actual economy versus its 2018-2019 growth trend

In [22]:
long = idx.melt(id_vars=["country", "year"], value_vars=["gdp_index", "trend_index"], var_name="path", value_name="index")
long["path"] = long["path"].map({"gdp_index": "Actual", "trend_index": "2018-19 growth trend (estimate)"})
fig = px.line(long, x="year", y="index", color="path", line_dash="path", facet_col="country", facet_col_wrap=4,
              category_orders={"country": COUNTRIES},
              color_discrete_map={"Actual": "#0072B2", "2018-19 growth trend (estimate)": "#9CA3AF"},
              line_dash_map={"Actual": "solid", "2018-19 growth trend (estimate)": "dash"},
              labels={"index": "", "year": ""})
fig.update_traces(line_width=2.8)
fig.update_xaxes(dtick=2, title=None); fig.update_yaxes(title=None)
polish(fig, "Actual economy versus the path implied by 2018-2019 growth",
       "Size of the economy, 2019 = 100; the gap between the lines is an estimate of the cost of the pandemic", height=560, facets=True)
FIGS["chart_08_actual_vs_trend"] = fig; fig.show()

**What it reveals.** By 2024, relative to the 2018-2019 trend, Rwanda's economy was approximately 11% below its projected level, China's 7%, Germany's and France's 5% each, the United States' 2% and India's 1%, while Brazil's exceeded its projected level by approximately 3%. The projected path is an estimate based on two years of pre-pandemic data.

---
# Theme 4. Jobs and prices
*How did unemployment and inflation move?*

### Chart 9. Unemployment rose most in the United States in 2020

In [23]:
chg = (u[2020] - u[2019]).rename("change").reset_index().sort_values("change")
chg["direction"] = np.where(chg["change"] >= 0, "Increase", "Decrease")
fig = px.bar(chg, x="change", y="country", orientation="h", color="direction",
             color_discrete_map={"Increase": "#D55E00", "Decrease": "#0072B2"},
             labels={"change": "Percentage points", "country": ""})
fig.update_traces(texttemplate="%{x:+.1f}", textposition="outside", cliponaxis=False)
fig.add_vline(x=0, line_color=INK)
fig.update_yaxes(categoryorder="array", categoryarray=list(chg["country"]))
polish(fig, "Change in the unemployment rate between 2019 and 2020",
       "Percentage points; the United States rose from 3.7% to 8.1%", height=440, legend=False)
FIGS["chart_09_unemployment_change"] = fig; fig.show()

**What it reveals.** The United States recorded by far the largest increase (4.4 points, from 3.7% to 8.1%), followed by Brazil (1.8) and India (1.4). Germany, Rwanda and China changed by less than one point, and France recorded a slight decline of 0.4. The United States was back to 3.6% by 2022, whereas Rwanda's unemployment rose later, from 11.8% in 2020 to 15.8% in 2021.

### Chart 10. Inflation rose in 2021-2022 and fell back by 2024

In [24]:
fig = px.line(eco, x="year", y="inflation_pct", color="country", facet_col="country", facet_col_wrap=4,
              markers=True, color_discrete_map=COUNTRY_COLORS, category_orders={"country": COUNTRIES},
              labels={"inflation_pct": "", "year": ""})
fig.update_yaxes(matches=None, showticklabels=True, title=None)
fig.update_xaxes(dtick=2, title=None)
fig.update_traces(line_width=2.8, marker_size=7)
fig.add_vrect(x0=2020.5, x1=2022.5, fillcolor="rgba(230,159,0,0.13)", line_width=0, layer="below", row="all", col="all")
polish(fig, "Annual inflation by country (%)",
       "Each panel has its own vertical scale; shaded band = 2021-2022", height=560, legend=False, facets=True)
FIGS["chart_10_inflation"] = fig; fig.show()

**What it reveals.** Inflation was low in 2020 (0.1% in Germany and 0.5% in France) and rose markedly in 2021-2022. In 2022 it reached 9.3% in Brazil, 8.0% in the United States and 6.9% in Germany, while Rwanda recorded 17.7% in 2022 and 19.8% in 2023, by far the highest values. China remained below 3% throughout. By 2024 inflation had fallen back to between 0.2% (China) and 5.0% (India). The chart shows when prices rose, not what caused the increase.

---
# Theme 5. Linking health, policy and the economy
*Does the economic loss follow the death toll or the strictness of restrictions?*

The economic measure used here is `gdp_effect`: GDP growth minus the country's own 2018-2019 average, in percentage points. Negative values mean growth fell below that country's normal.

### Chart 11. In 2020 the economic loss did not follow the reported death toll

In [25]:
c20 = cov[cov.year == 2020].copy()
POS = {"Brazil": "top center", "United States": "bottom center", "Germany": "middle left",
       "France": "top center", "India": "top center", "Rwanda": "top center", "China": "top center"}
fig = px.scatter(c20, x="new_deaths_per_million", y="gdp_effect", color="country", text="country", log_x=True,
                 color_discrete_map=COUNTRY_COLORS,
                 labels={"new_deaths_per_million": "Reported deaths per million, 2020 (logarithmic scale)",
                         "gdp_effect": "GDP growth minus 2018-19 average (points)"})
fig.update_traces(marker=dict(size=16, line=dict(width=1.5, color="white")), textfont_size=12)
fig.for_each_trace(lambda t: t.update(textposition=POS.get(t.name, "top center")))
fig.add_hline(y=0, line_color=INK)
polish(fig, "Economic loss versus reported deaths, 2020",
       "Each dot is one country; lower on the chart means a larger fall below its own normal growth", height=520, legend=False)
FIGS["chart_11_loss_vs_deaths_2020"] = fig; fig.show()

**What it reveals.** The largest economic losses in 2020 were in Rwanda (-11.5 points) and India (-10.9), which reported very few deaths (6 and 104 per million). France (-9.3), Germany (-5.2), the United States (-4.9) and Brazil (-4.8) reported 540 to 1,009 deaths per million and lost less relative to their own normal. China reported almost no deaths and lost 4.1 points. There is no visible relationship between the two measures (Spearman +0.07, 7 countries).

### Chart 12. Restriction strictness and economic impact, 2020-2022

In [26]:
fig = px.scatter(cov, x="sti_weighted", y="gdp_effect", color="country", text="year_str",
                 color_discrete_map=COUNTRY_COLORS,
                 labels={"sti_weighted": "Stringency index (yearly average, 0-100)",
                         "gdp_effect": "GDP growth minus 2018-19 average (points)"})
fig.update_traces(marker=dict(size=13, line=dict(width=1, color="white")), textposition="top center", textfont_size=10)
sub = cov[["sti_weighted", "gdp_effect"]].dropna()
slope, intercept = np.polyfit(sub.sti_weighted, sub.gdp_effect, 1)
xs = np.linspace(sub.sti_weighted.min(), sub.sti_weighted.max(), 50)
fig.add_trace(go.Scatter(x=xs, y=intercept + slope * xs, mode="lines", name="Fitted line (all points)",
                         line=dict(color=INK, dash="dot", width=2)))
fig.add_hline(y=0, line_color="#9CA3AF")
polish(fig, "Restriction strictness versus economic impact, 2020-2022",
       "21 points = 7 countries x 3 years; the label on each dot is the year", height=540)
FIGS["chart_12_strictness_vs_loss"] = fig; fig.show()

**What it reveals.** No clear pattern is visible (Spearman -0.10, p = 0.68). The points separate by year more than by strictness: 2020 lies below zero for every country, 2021 above zero and 2022 close to zero. The fitted line summarizes the overall tendency and does not indicate cause.

## 6. Statistical tests

Spearman rank correlation is used because it compares rankings, so extreme values (for example China's near-zero case counts) do not distort it. The coefficient runs from -1 to +1, and a p-value below 0.05 indicates a pattern unlikely to be due to chance. The sample is small (21 country-years from 7 countries, each appearing three times), so the p-values are optimistic and the results are exploratory.

In [27]:
def spear(df, x, yv):
    s = df[[x, yv]].dropna()
    rho, p = spearmanr(s[x], s[yv])
    return rho, p, len(s)

NAMES = {"sti_weighted": "Restriction strictness", "new_deaths_per_million": "Reported deaths",
         "new_cases_per_million": "Reported cases", "gdp_effect": "GDP effect",
         "unemp_effect": "Unemployment effect", "infl_effect": "Inflation effect"}
pairs = [("sti_weighted", "gdp_effect"), ("new_deaths_per_million", "gdp_effect"), ("new_cases_per_million", "gdp_effect"),
         ("sti_weighted", "unemp_effect"), ("new_deaths_per_million", "unemp_effect"), ("sti_weighted", "infl_effect")]
rows = []
for a, b in pairs:
    rho, p, n = spear(cov, a, b)
    strength = "very weak" if abs(rho) < 0.1 else "weak" if abs(rho) < 0.3 else "moderate" if abs(rho) < 0.5 else "strong"
    rows.append({"Relationship": f"{NAMES[a]} vs {NAMES[b]}", "rho": rho, "p-value": p, "n": n,
                 "Reading": f"{strength} {'positive' if rho > 0 else 'negative'}, " + ("significant at 5%" if p < 0.05 else "not significant")})
results = pd.DataFrame(rows).set_index("Relationship")
display(results.style.format({"rho": "{:+.2f}", "p-value": "{:.3f}", "n": "{:d}"})
        .map(lambda v: "background-color:#FDE68A;font-weight:600" if isinstance(v, float) and v < 0.05 else "", subset=["p-value"])
        .set_table_styles([{"selector": "th", "props": [("background-color", "#F3F4F6"), ("color", INK), ("font-weight", "600")]}]))

,rho,p-value,n,Reading
Relationship,,,,
Restriction strictness vs GDP effect,-0.09,0.683,21,"very weak negative, not significant"
Reported deaths vs GDP effect,+0.21,0.351,21,"weak positive, not significant"
Reported cases vs GDP effect,+0.32,0.156,21,"moderate positive, not significant"
Restriction strictness vs Unemployment effect,+0.34,0.128,21,"moderate positive, not significant"
Reported deaths vs Unemployment effect,+0.05,0.836,21,"very weak positive, not significant"
Restriction strictness vs Inflation effect,-0.51,0.019,21,"strong negative, significant at 5%"


**Robustness check:** the two main relationships, year by year (only 7 countries per year, so this is a rough guide).

In [28]:
rob = []
for yr in (2020, 2021, 2022):
    s = cov[cov.year == yr]
    for a in ("sti_weighted", "new_deaths_per_million"):
        rho, p, n = spear(s, a, "gdp_effect")
        rob.append({"Year": yr, "Relationship": f"{NAMES[a]} vs GDP effect", "rho": rho, "p-value": p, "n": n})
rob = pd.DataFrame(rob).set_index(["Year", "Relationship"])
display(rob.style.format({"rho": "{:+.2f}", "p-value": "{:.3f}", "n": "{:d}"}))

## 7. Evidence-based findings for the five themes

The text below is **generated from the data** by the code in this cell, so it always matches the tables and charts above.

In [29]:
def mon(ts): return pd.Timestamp(ts).strftime("%B %Y")
fmt0 = lambda v: f"{v:,.0f}"
nm = lambda c: "the United States" if c == "United States" else c          # reads naturally in a sentence
Nm = lambda c: "The United States" if c == "United States" else c

# ---- Theme 1: health ----
dt = y.groupby("country")["new_deaths_per_million"].sum().sort_values(ascending=False)
pk = m.loc[m.groupby("country")["new_deaths_per_million"].idxmax()].sort_values("date")
f1 = [
 f"Reported deaths over 2020-2022 were highest in {nm(dt.index[0])} ({fmt0(dt.iloc[0])} per million) and {nm(dt.index[1])} ({fmt0(dt.iloc[1])}), and lowest in {nm(dt.index[-1])} ({fmt0(dt.iloc[-1])}) and {nm(dt.index[-2])} ({fmt0(dt.iloc[-2])}).",
 f"The deadliest month ranged from {mon(pk.iloc[0]['date'])} ({nm(pk.iloc[0]['country'])}) to {mon(pk.iloc[-1]['date'])} ({nm(pk.iloc[-1]['country'])}): the waves were not synchronized across countries.",
 "Reported figures depend on testing and registration, so low values in some countries do not prove a milder epidemic."]

# ---- Theme 2: policy ----
pks = m.loc[m.groupby("country")["sti_weighted"].idxmax()]
n_spring = int((pks["date"].dt.year == 2020).sum())
ym = y.pivot(index="country", columns="year", values="sti_weighted")
rose = [c for c in ym.index if ym.loc[c, 2022] > ym.loc[c, 2020]]
gp = m.loc[m.groupby("country")["sti_gap"].idxmax()].sort_values("sti_gap", ascending=False)
no_gap = [c for c in COUNTRIES if m[m.country == c]["sti_gap"].max() < 0.01]
f2 = [
 f"{n_spring} of 7 countries reached their strictest month during 2020; the highest stringency recorded was {pks['sti_weighted'].max():.1f} ({nm(pks.loc[pks['sti_weighted'].idxmax(), 'country'])}, {mon(pks.loc[pks['sti_weighted'].idxmax(), 'date'])}).",
 f"Average stringency fell between 2020 and 2022 in {7 - len(rose)} countries and rose only in {', '.join(rose) if rose else 'none'} ({ym.loc[rose[0], 2020]:.1f} to {ym.loc[rose[0], 2022]:.1f}).",
 f"The largest gap between the rules for non-vaccinated and vaccinated people was {gp.iloc[0]['sti_gap']:.1f} points in {nm(gp.iloc[0]['country'])} ({mon(gp.iloc[0]['date'])}), followed by {nm(gp.iloc[1]['country'])} ({gp.iloc[1]['sti_gap']:.1f}); {', '.join(nm(c) for c in no_gap)} recorded no gap."]

# ---- Theme 3: economy ----
g = eco.pivot(index="country", columns="year", values="gdp_growth_pct")
med_g = g.median()
n_neg = int((g[2020] < 0).sum())
ge = full[full.year == 2020].set_index("country")["gdp_effect"].sort_values()
i21 = idx[idx.year == 2021].set_index("country")["gdp_index"]
below21 = list(i21[i21 < 100].index)
gap24 = idx[idx.year == 2024].set_index("country")["gap_pct"].sort_values()
f3 = [
 f"Median GDP growth fell from {med_g[2019]:.1f}% in 2019 to {med_g[2020]:.1f}% in 2020 and rebounded to {med_g[2021]:.1f}% in 2021; {n_neg} of 7 economies contracted in 2020.",
 f"Relative to their own 2018-2019 average, the largest falls in 2020 were in {nm(ge.index[0])} ({ge.iloc[0]:.1f} points), {nm(ge.index[1])} ({ge.iloc[1]:.1f}) and {nm(ge.index[2])} ({ge.iloc[2]:.1f}).",
 f"In 2021, {' and '.join(nm(c) for c in below21)} were still below their 2019 size; by 2022 all economies were above it.",
 f"In 2024, {int((gap24 < 0).sum())} of 7 economies remained below the estimated 2018-2019 growth trend, the largest shortfall being {nm(gap24.index[0])} ({gap24.iloc[0]:.0f}%)."]

# ---- Theme 4: jobs and prices ----
uc = (u[2020] - u[2019]).sort_values(ascending=False)
inf = eco.pivot(index="country", columns="year", values="inflation_pct")
imax = inf.stack().idxmax()
f4 = [
 f"Unemployment rose most between 2019 and 2020 in {nm(uc.index[0])} ({uc.iloc[0]:+.1f} points, from {u.loc[uc.index[0], 2019]:.1f}% to {u.loc[uc.index[0], 2020]:.1f}%) and fell in {', '.join(nm(c) for c in uc[uc < 0].index)}.",
 f"Median inflation was {inf[2020].median():.1f}% in 2020 and {inf[2022].median():.1f}% in 2022: prices rose after the output and jobs shock, not during it.",
 f"The highest inflation recorded was {inf.loc[imax]:.1f}% ({nm(imax[0])}, {imax[1]}); {', '.join(nm(c) for c in COUNTRIES if inf.loc[c].max() < 3)} stayed below 3% throughout."]

# ---- Theme 5: links ----
sig = results[results["p-value"] < 0.05]
r20 = spear(cov[cov.year == 2020], "new_deaths_per_million", "gdp_effect")
d20 = y[y.year == 2020].set_index("country")["new_deaths_per_million"]
f5 = [
 f"In 2020, the two countries with the largest economic loss ({nm(ge.index[0])} and {nm(ge.index[1])}) reported {d20[ge.index[0]]:,.0f} and {d20[ge.index[1]]:,.0f} deaths per million, far below France ({d20['France']:,.0f}) or the United States ({d20['United States']:,.0f}); the rank correlation between deaths and economic loss is {r20[0]:+.2f} (n = {r20[2]}).",
 f"Across 2020-2022, {len(sig)} of {len(results)} correlation tests reached p < 0.05" + (f" ({', '.join(sig.index)})." if len(sig) else "."),
 "Restrictions eased across countries in 2022, the same year inflation peaked, so a link between strictness and inflation reflects timing and does not indicate cause."]

THEMES = [("1. Health burden", f1), ("2. Government response", f2), ("3. Economic output and recovery", f3),
          ("4. Jobs and prices", f4), ("5. Linking health, policy and the economy", f5)]
for title, items in THEMES:
    display(Markdown(f"### {title}\n" + "\n".join(f"- {t}" for t in items)))

### 1. Health burden
- Reported deaths over 2020-2022 were highest in Brazil (3,296 per million) and the United States (3,174), and lowest in China (31) and Rwanda (107).
- The deadliest month ranged from November 2020 (France) to December 2022 (China): the waves were not synchronized across countries.
- Reported figures depend on testing and registration, so low values in some countries do not prove a milder epidemic.

### 2. Government response
- 6 of 7 countries reached their strictest month during 2020; the highest stringency recorded was 98.6 (India, April 2020).
- Average stringency fell between 2020 and 2022 in 6 countries and rose only in China (68.3 to 77.6).
- The largest gap between the rules for non-vaccinated and vaccinated people was 45.4 points in France (September 2021), followed by Rwanda (44.5); China recorded no gap.

### 3. Economic output and recovery
- Median GDP growth fell from 2.6% in 2019 to -3.3% in 2020 and rebounded to 6.9% in 2021; 6 of 7 economies contracted in 2020.
- Relative to their own 2018-2019 average, the largest falls in 2020 were in Rwanda (-11.5 points), India (-10.9) and France (-9.3).
- In 2021, France and Germany were still below their 2019 size; by 2022 all economies were above it.
- In 2024, 6 of 7 economies remained below the estimated 2018-2019 growth trend, the largest shortfall being Rwanda (-11%).

### 4. Jobs and prices
- Unemployment rose most between 2019 and 2020 in the United States (+4.4 points, from 3.7% to 8.1%) and fell in France.
- Median inflation was 2.4% in 2020 and 6.9% in 2022: prices rose after the output and jobs shock, not during it.
- The highest inflation recorded was 19.8% (Rwanda, 2023); China stayed below 3% throughout.

### 5. Linking health, policy and the economy
- In 2020, the two countries with the largest economic loss (Rwanda and India) reported 6 and 104 deaths per million, far below France (975) or the United States (1,009); the rank correlation between deaths and economic loss is +0.07 (n = 7).
- Across 2020-2022, 1 of 6 correlation tests reached p < 0.05 (Restriction strictness vs Inflation effect).
- Restrictions eased across countries in 2022, the same year inflation peaked, so a link between strictness and inflation reflects timing and does not indicate cause.

## 8. Limits of the analysis (for the critical-analysis section)

- **Small, selected sample.** Seven countries chosen by the team, not a sample of the world; results describe these countries only.
- **Short health and policy window.** Health and policy data cover 2020-2022, so links with the economy rest on 21 country-years.
- **Short baseline.** The pre-COVID reference is two years (2018-2019), so baselines and the trend path are estimates.
- **Reporting differences.** Case and death counts depend on testing and registration rules and are not fully comparable across countries.
- **Association, not cause.** Correlations do not show that restrictions or deaths caused economic outcomes; the 2021 rebound is partly a base effect, and the 2022 price rise overlaps with other events.

## 9. Export: findings document, tables and chart files

In [30]:
os.makedirs("outputs/charts", exist_ok=True)

lines = ["# Findings summary: COVID-19 and the economy",
         "*Brazil, China, France, Germany, India, Rwanda and the United States. Economy 2018-2024; health and policy 2020-2022.*", ""]
for title, items in THEMES:
    lines += [f"## {title}"] + [f"- {t}" for t in items] + [""]
lines += ["## Limits",
          "- Seven selected countries; health and policy data cover 2020-2022 only (21 country-years).",
          "- The pre-COVID baseline and trend are estimates based on 2018-2019.",
          "- Reported cases and deaths are not fully comparable across countries.",
          "- Associations do not establish cause."]
with open("outputs/findings_summary.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

for name, tbl in {"table1_typical_economy_by_period": t1, "table2_economy_by_country": t2, "table3_health_policy_by_country": t3,
                  "table4_economic_impact": impact, "spearman_results": results}.items():
    tbl.to_csv(f"outputs/{name}.csv")

for name, fg in FIGS.items():
    fg.write_json(f"outputs/charts/{name}.json")            # chart specification
    fg.write_html(f"outputs/charts/{name}.html", include_plotlyjs="cdn")   # standalone interactive file

print("Saved in ./outputs :", sorted(os.listdir("outputs")))
print("Charts             :", len(os.listdir("outputs/charts")), "files")
display(Markdown("\n".join(lines)))

Saved in ./outputs : ['charts', 'findings_summary.md', 'spearman_results.csv', 'table1_typical_economy_by_period.csv', 'table2_economy_by_country.csv', 'table3_health_policy_by_country.csv', 'table4_economic_impact.csv']
Charts             : 24 files


# Findings summary: COVID-19 and the economy
*Brazil, China, France, Germany, India, Rwanda and the United States. Economy 2018-2024; health and policy 2020-2022.*

## 1. Health burden
- Reported deaths over 2020-2022 were highest in Brazil (3,296 per million) and the United States (3,174), and lowest in China (31) and Rwanda (107).
- The deadliest month ranged from November 2020 (France) to December 2022 (China): the waves were not synchronized across countries.
- Reported figures depend on testing and registration, so low values in some countries do not prove a milder epidemic.

## 2. Government response
- 6 of 7 countries reached their strictest month during 2020; the highest stringency recorded was 98.6 (India, April 2020).
- Average stringency fell between 2020 and 2022 in 6 countries and rose only in China (68.3 to 77.6).
- The largest gap between the rules for non-vaccinated and vaccinated people was 45.4 points in France (September 2021), followed by Rwanda (44.5); China recorded no gap.

## 3. Economic output and recovery
- Median GDP growth fell from 2.6% in 2019 to -3.3% in 2020 and rebounded to 6.9% in 2021; 6 of 7 economies contracted in 2020.
- Relative to their own 2018-2019 average, the largest falls in 2020 were in Rwanda (-11.5 points), India (-10.9) and France (-9.3).
- In 2021, France and Germany were still below their 2019 size; by 2022 all economies were above it.
- In 2024, 6 of 7 economies remained below the estimated 2018-2019 growth trend, the largest shortfall being Rwanda (-11%).

## 4. Jobs and prices
- Unemployment rose most between 2019 and 2020 in the United States (+4.4 points, from 3.7% to 8.1%) and fell in France.
- Median inflation was 2.4% in 2020 and 6.9% in 2022: prices rose after the output and jobs shock, not during it.
- The highest inflation recorded was 19.8% (Rwanda, 2023); China stayed below 3% throughout.

## 5. Linking health, policy and the economy
- In 2020, the two countries with the largest economic loss (Rwanda and India) reported 6 and 104 deaths per million, far below France (975) or the United States (1,009); the rank correlation between deaths and economic loss is +0.07 (n = 7).
- Across 2020-2022, 1 of 6 correlation tests reached p < 0.05 (Restriction strictness vs Inflation effect).
- Restrictions eased across countries in 2022, the same year inflation peaked, so a link between strictness and inflation reflects timing and does not indicate cause.

## Limits
- Seven selected countries; health and policy data cover 2020-2022 only (21 country-years).
- The pre-COVID baseline and trend are estimates based on 2018-2019.
- Reported cases and deaths are not fully comparable across countries.
- Associations do not establish cause.